In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10 

In [5]:
#DATASETS and DATALoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainSet=CIFAR10(root="./data2",train=True,download=True,transform=transform)
testSet=CIFAR10(root="./data2",train=False,download=True,transform=transform)



100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [00:33<00:00, 5.10MB/s]


In [7]:
trainSet

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data2
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [8]:
trainloader=DataLoader(trainSet,batch_size=64,shuffle=True)
testloader=DataLoader(testSet,batch_size=64)

In [25]:
#Build the CNNc
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
           nn.Conv2d(3, 32, kernel_size=3, padding=1),
           nn.ReLU(),
           nn.MaxPool2d(2, 2),

           nn.Conv2d(32, 64, kernel_size=3, padding=1),
           nn.ReLU(),
           nn.MaxPool2d(2, 2),

           nn.Conv2d(64, 128, kernel_size=3, padding=1),
           nn.ReLU(),
           nn.MaxPool2d(2, 2)
        )
        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )
    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x

In [26]:
model=CNN()

In [27]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [35]:
#training the CNN
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    val_loss=0.0
    
    for images,labels in trainloader:
        optimizer.zero_grad()
        output =model.forward(images)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss+=loss.item()
    #validation loss calculate
    model.eval()
    with torch.no_grad():
        for images,labels in testloader:
            outputs=model(images)
            loss=criterion(outputs,labels)
            val_loss+=loss.item()
        avg_validate=val_loss/len(testloader)

        print(f"validation Loss = {avg_validate}")
    
    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

validation Loss = 1.4964063513051173
epoch=1/10 & loss=0.09155112008214035
validation Loss = 1.4774290532063528
epoch=2/10 & loss=0.0840369013678211
validation Loss = 1.561064781277043
epoch=3/10 & loss=0.08116278348429858
validation Loss = 1.619907315559448
epoch=4/10 & loss=0.07610169351916722
validation Loss = 1.727901282583832
epoch=5/10 & loss=0.07135635870092018
validation Loss = 1.6900381628115466
epoch=6/10 & loss=0.06672365351459797
validation Loss = 1.7621377419893909
epoch=7/10 & loss=0.06308073566585203
validation Loss = 1.7783627217742288
epoch=8/10 & loss=0.07369400373415289
validation Loss = 1.9155044081104788
epoch=9/10 & loss=0.0617711141799598
validation Loss = 1.9031676205859822
epoch=10/10 & loss=0.06557117712535404


In [34]:
#evaluate our model

correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images,labels in testloader: 
        outputs =model.forward(images)
        _, predicted = torch.max(outputs,1)

        correct_labels = correct_labels + (predicted == labels).sum().item()
        total_labels +=labels.size(0)
print(f"accuracy = {correct_labels/total_labels*100}")


accuracy = 74.35000000000001
